# Tutorial: Dos agentes hablando con A2A

**KEEPCODING · AI Engineering · Clase 5 · Módulo M3: Agent 2 Agent (A2A)**

En este laboratorio levantamos **dos agentes** y los hacemos colaborar usando el protocolo **A2A** (Agent2Agent). Todo ocurre dentro de este mismo notebook: un agente actúa de servidor y el otro de cliente.

## Qué necesitas

- Una clave `OPENAI_API_KEY`. El agente especialista la usa para redactar su respuesta final con un LLM.
- Este notebook corre en **Google Colab** o en **Jupyter local** sin cambios.
- Conexión a internet: consultamos la API pública de One Piece y, si falla, usamos un diccionario de respaldo.

## Para quién es

Alumnado del curso que ya conoce Python básico, async/await y la idea de "un agente con tools". No hace falta saber A2A: lo construimos paso a paso.

## Qué vas a aprender

Al terminar sabrás explicar y reproducir el **handshake de A2A**:

1. **Discovery**: un agente publica su *Agent Card* y otro la lee.
2. **Delegación**: el orquestador envía una tarea al especialista.
3. **Respuesta**: el especialista devuelve un *artifact* y el orquestador lo recibe.

El hilo conductor es divertido: el agente especialista es un **experto en One Piece**.

### Por qué importa

MCP conecta un agente con tools; **A2A conecta agentes entre sí**. Es el protocolo que permite que sistemas multiagente (incluso de distintos vendors) colaboren sin escribir integraciones a medida.

### Conecta con

- **Slides M3** · Agent Card, discovery, delegación de tareas.
- **M1 / M2** · los agentes que ahora aprenden a hablar entre sí.
- **M1 / Lab 3 (MCP)** · el protocolo complementario (tools vs agentes).
- **M5** · A2A aparece en los protocolos soportados del `AGENTS.md`.

### Stack

`a2a-sdk==1.0.3` · `uvicorn` · `httpx` · `openai` · `gpt-5-mini`.

### Hardware

Cualquier CPU; Colab Free sobra. Los servidores corren en hilos del propio notebook (con health-check al arrancar y apagado limpio al final).


## Outline

1. Setup: instalar el SDK de A2A y preparar el entorno async del notebook.
2. Construir la skill del especialista: consultar personajes de One Piece.
3. Envolver al especialista como **servidor A2A** y levantarlo en `localhost:8001`.
4. Leer la **Agent Card** en `/.well-known/agent-card.json`.
5. Crear el **orquestador** (cliente A2A) y delegar una tarea.
6. Recibir la respuesta del especialista (el artifact).
7. (Opcional) Una tarea que pide más datos: `input_required` y human-in-the-loop.
8. Recap del handshake completo.
9. Apagar los servidores locales.

## 1. Setup

### Qué es A2A en una frase

**A2A (Agent2Agent)** es un protocolo abierto para que agentes de IA, construidos por equipos distintos y con frameworks distintos, **se descubran y colaboren** sin acoplarse entre sí.

Las piezas clave que usaremos:

- **Agent Card**: un JSON público (en `/.well-known/agent-card.json`) que describe el agente: nombre, descripción, skills y endpoint. Es la "tarjeta de visita".
- **Task**: una unidad de trabajo que un agente delega a otro.
- **Artifact**: el resultado que el agente produce y devuelve.
- **AgentExecutor**: la clase donde escribimos la lógica del agente servidor.

Instalamos el SDK oficial de Python (`a2a-sdk`) con su servidor HTTP, más las utilidades para correr todo dentro del notebook.

In [1]:
# Instalación silenciosa. En Colab puede tardar ~1 minuto.
# a2a-sdk fijado a 1.0.3: la v1.0 fue una reescritura mayor de la API.
# Sin fijar la versión, una futura release con cambios de API rompería el lab.
%pip install -q "a2a-sdk[http-server]==1.0.3" uvicorn httpx nest_asyncio openai
print("Dependencias instaladas")

Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas


### Async dentro del notebook

Un notebook ya corre dentro de un *event loop* de asyncio. Para poder, además, levantar un servidor HTTP async **dentro** de ese loop necesitamos `nest_asyncio`: permite anidar loops.

El servidor lo lanzaremos más adelante en un **hilo de fondo** para que el notebook siga respondiendo mientras el agente escucha peticiones.

In [2]:
import asyncio
import os
import threading
import time
import uuid

import httpx
import nest_asyncio
import uvicorn

# Permite anidar event loops: imprescindible para correr un server async aquí.
nest_asyncio.apply()

print("Entorno async listo")

Entorno async listo


### Clave de OpenAI

El especialista usa un LLM para redactar la respuesta final con un tono simpático. Si estás en Colab, usa el panel de *Secrets* (icono de la llave) o introduce la clave cuando se te pida.

In [3]:
# Cargamos OPENAI_API_KEY. Si no está en el entorno, la pedimos por teclado.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        # En Colab: lee el secret guardado en el panel lateral.
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass

        os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce tu OPENAI_API_KEY: ")

assert os.environ.get("OPENAI_API_KEY"), "Falta OPENAI_API_KEY"
print("OPENAI_API_KEY cargada")

OPENAI_API_KEY cargada


## 2. La skill del especialista: experto en One Piece

Antes de hablar de A2A, escribimos la **lógica de negocio** del agente especialista: una función que, dado el nombre de un personaje, devuelve información sobre él.

Estrategia robusta para clase:

1. Primero consulta la **API pública de One Piece** (`api-onepiece.com`).
2. Si la API falla (sin internet, caída, etc.) usa un **diccionario de respaldo** hardcodeado.

Así el laboratorio funciona siempre, con o sin red.

In [4]:
import unicodedata

# Diccionario de respaldo: datos mínimos por si la API no responde.
ONE_PIECE_FALLBACK = {
    "luffy": {
        "name": "Monkey D. Luffy",
        "crew": "Piratas de Sombrero de Paja",
        "bounty": "3.000.000.000 berries",
        "fruit": "Gomu Gomu no Mi (Hito Hito no Mi, modelo Nika)",
        "dream": "Convertirse en el Rey de los Piratas",
    },
    "zoro": {
        "name": "Roronoa Zoro",
        "crew": "Piratas de Sombrero de Paja",
        "bounty": "1.111.000.000 berries",
        "fruit": "Ninguna",
        "dream": "Ser el mejor espadachín del mundo",
    },
    "nami": {
        "name": "Nami",
        "crew": "Piratas de Sombrero de Paja",
        "bounty": "366.000.000 berries",
        "fruit": "Ninguna",
        "dream": "Dibujar un mapa de todo el mundo",
    },
}

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE = []


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden.

    El LLM puede pasar "Luffy" o "Monkey D. Luffy"; la API guarda
    "Monkey D Luffy" (sin punto). Normalizamos ambos y comparamos por
    contención o por palabras, nunca con un substring frágil.
    """
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op() -> list:
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    import urllib.request
    import json as _json
    url = "https://api.api-onepiece.com/v2/characters/en"
    req = urllib.request.Request(url, headers={"User-Agent": "keepcoding-ai-lab/1.0"})
    with urllib.request.urlopen(req, timeout=25) as r:
        _OP_CACHE = _json.load(r)
    return _OP_CACHE


def consultar_one_piece_api(nombre: str) -> dict | None:
    """Consulta la API pública de One Piece. Devuelve None si falla."""
    try:
        for p in _cargar_personajes_op():
            if _coincide(nombre, p.get("name") or ""):
                return {
                    "name": p.get("name", nombre),
                    "crew": (p.get("crew") or {}).get("name", "Desconocida"),
                    "bounty": p.get("bounty") or "Desconocida",
                    "fruit": (p.get("fruit") or {}).get("name", "Ninguna"),
                    "fuente": "api-onepiece.com",
                }
    except Exception as e:
        print(f"  [aviso] API de One Piece no disponible: {e}")
    return None


def buscar_personaje(nombre: str) -> dict:
    """Devuelve datos de un personaje: API primero, diccionario de respaldo después."""
    datos = consultar_one_piece_api(nombre)
    if datos:
        return datos
    for clave, info in ONE_PIECE_FALLBACK.items():
        if _coincide(nombre, clave) or _coincide(nombre, info["name"]):
            datos = dict(info)
            datos["fuente"] = "diccionario de respaldo"
            return datos
    return {"name": nombre, "fuente": "no encontrado",
            "nota": "No tengo datos de este personaje."}


# Prueba rápida de la skill, todavía sin A2A.
buscar_personaje("Luffy")

{'name': 'Monkey D Luffy',
 'crew': 'The Chapeau de Paille crew',
 'bounty': '3.000.000.000',
 'fruit': 'Hito Hito no Mi, Nika model',
 'fuente': 'api-onepiece.com'}

### El especialista redacta con un LLM

`buscar_personaje` devuelve datos crudos. El agente especialista los pasa por un LLM para responder con un tono ameno, como un auténtico fan de One Piece. Esto separa **datos** (la tool) de **redacción** (el LLM): un patrón habitual en agentes.

In [5]:
from openai import OpenAI

openai_client = OpenAI()


def redactar_respuesta(pregunta: str, datos: dict) -> str:
    """Convierte datos crudos en una respuesta natural usando un LLM."""
    prompt = (
        "Eres un experto entusiasta en el anime One Piece. "
        "Responde en español, breve y simpático, usando solo estos datos. No uses rayas largas (em dash):\n"
        f"{datos}\n\n"
        f"Pregunta del usuario: {pregunta}"
    )
    # gpt-5-mini solo admite la temperatura por defecto, así que no la fijamos.
    respuesta = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return respuesta.choices[0].message.content.strip()


# Nota: esta celda define la función. La probaremos a través del agente A2A.
print("Redactor del especialista listo")

Redactor del especialista listo


## 3. Envolver al especialista como servidor A2A

Ahora viene la parte de A2A. Para publicar un agente como servidor necesitamos dos cosas:

1. **Un `AgentExecutor`**: la clase que ejecuta la tarea. Recibe el input del cliente, hace el trabajo y publica el resultado como *artifact*.
2. **Una `AgentCard`**: la tarjeta de visita pública con nombre, descripción, skills y endpoint.

Empezamos por el `AgentExecutor`. El método `execute` recibe un `RequestContext` (con el input del cliente) y un `EventQueue` (por donde emite eventos: inicio, artifact, fin).

In [6]:
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.tasks import TaskUpdater
from a2a.helpers import new_task_from_user_message
from a2a.types import Part, TaskState


def extraer_nombre_personaje(texto: str) -> str:
    """Heurística simple: toma la última palabra con mayúscula del mensaje."""
    candidatos = [w.strip(".,!?") for w in texto.split() if w[:1].isupper()]
    return candidatos[-1] if candidatos else texto.strip()


class ExpertoOnePieceExecutor(AgentExecutor):
    """Agente especialista: responde preguntas sobre personajes de One Piece."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # En A2A el agente debe publicar primero la Task; después los updates.
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)

        # TaskUpdater es el ayudante para emitir eventos de la tarea.
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()  # estado: working

        # 1. Leemos lo que pide el cliente.
        pregunta = context.get_user_input()
        print(f"  [especialista] tarea recibida: {pregunta!r}")

        # 2. Ejecutamos la skill: datos + redacción con LLM.
        nombre = extraer_nombre_personaje(pregunta)
        datos = buscar_personaje(nombre)
        texto = redactar_respuesta(pregunta, datos)

        # 3. Publicamos el resultado como artifact y cerramos la tarea.
        # En a2a 1.0 un Part lleva el texto directamente: Part(text=...).
        await updater.add_artifact(
            [Part(text=texto)],
            name="respuesta_one_piece",
        )
        await updater.complete()  # estado: completed
        print("  [especialista] tarea completada")

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task_from_user_message(context.message)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.update_status(TaskState.TASK_STATE_CANCELED)


print("AgentExecutor del especialista definido")

AgentExecutor del especialista definido


### La Agent Card del especialista

La `AgentCard` es lo que otros agentes leerán para decidir si este agente les sirve. Incluye:

- `name` y `description`: para que un orquestador entienda de qué va.
- `skills`: lista de habilidades. Cada `AgentSkill` tiene `id`, `name`, `description`, `tags` y ejemplos.
- `supported_interfaces`: los transportes y endpoints donde vive el agente.
- `capabilities`: por ejemplo si soporta streaming.

Definimos una skill llamada `consultar_personaje`.

In [7]:
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill

HOST = "127.0.0.1"
PORT = 8001
ESPECIALISTA_URL = f"http://{HOST}:{PORT}"

skill_consultar = AgentSkill(
    id="consultar_personaje",
    name="Consultar personaje de One Piece",
    description=(
        "Dado el nombre de un personaje de One Piece, devuelve información "
        "sobre su tripulación, recompensa y fruta del diablo."
    ),
    tags=["one-piece", "anime", "personajes"],
    examples=["Dame info de Luffy", "¿Qué sabes de Zoro?"],
)

# En a2a 1.0 la AgentCard ya no tiene un campo 'url' suelto: declara una lista
# de 'supported_interfaces', cada uno con su transporte (JSONRPC) y endpoint.
agent_card_especialista = AgentCard(
    name="Experto en One Piece",
    description="Agente especialista que responde preguntas sobre personajes de One Piece.",
    supported_interfaces=[
        AgentInterface(protocol_binding="JSONRPC", url=ESPECIALISTA_URL + "/"),
    ],
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    skills=[skill_consultar],
)

print(f"Agent Card creada: {agent_card_especialista.name}")
print(f"Skills: {[s.id for s in agent_card_especialista.skills]}")

Agent Card creada: Experto en One Piece
Skills: ['consultar_personaje']


### Construir la app del servidor

Juntamos las piezas:

- `DefaultRequestHandler`: enruta las peticiones A2A hacia nuestro `AgentExecutor`. Necesita un `TaskStore` (aquí en memoria) para guardar el estado de las tareas y la `AgentCard` del agente.
- Las funciones `create_agent_card_routes` y `create_jsonrpc_routes`: generan las rutas HTTP del protocolo A2A. La primera publica la Agent Card; la segunda expone el endpoint JSON-RPC por el que llegan las tareas.
- Montamos esas rutas en una app `Starlette`, que es el servidor HTTP async que correrá `uvicorn`.

In [8]:
from starlette.applications import Starlette

from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore

request_handler = DefaultRequestHandler(
    agent_executor=ExpertoOnePieceExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=agent_card_especialista,
)

# Rutas del protocolo A2A: la Agent Card pública y el endpoint JSON-RPC.
rutas = []
rutas.extend(create_agent_card_routes(agent_card_especialista))
rutas.extend(create_jsonrpc_routes(request_handler, rpc_url="/"))

server_app = Starlette(routes=rutas)

print("App del servidor A2A construida")

App del servidor A2A construida


### Levantar el servidor en un hilo de fondo

Si llamásemos a `uvicorn.run()` directamente, **bloquearía** el notebook. La solución: lanzar uvicorn en un **hilo de fondo** (`threading.Thread`). Así el servidor escucha en `localhost:8001` mientras seguimos ejecutando celdas.

In [9]:
# Guardamos el objeto Server para poder apagarlo limpiamente al final (sección 9).
servidor_especialista = uvicorn.Server(
    uvicorn.Config(server_app, host=HOST, port=PORT, log_level="warning")
)


def arrancar_servidor():
    """Corre uvicorn en este hilo. Se invoca desde un thread de fondo."""
    servidor_especialista.run()


server_thread = threading.Thread(target=arrancar_servidor, daemon=True)
server_thread.start()

# Esperamos a que el servidor RESPONDA (health-check), en vez de un sleep fijo:
# en Colab el arranque puede tardar más o menos, así evitamos una race condition.
import httpx as _httpx
def _esperar_servidor(url, intentos=40):
    for _ in range(intentos):  # hasta ~20s
        try:
            if _httpx.get(f"{url}/.well-known/agent-card.json", timeout=1.0).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False

if not _esperar_servidor(ESPECIALISTA_URL):
    raise RuntimeError("El servidor del especialista no respondió a tiempo (revisa la celda anterior).")
print(f"Servidor del especialista escuchando en {ESPECIALISTA_URL}")

Servidor del especialista escuchando en http://127.0.0.1:8001


## 4. Discovery: leer la Agent Card

El primer paso de A2A es el **descubrimiento**. Cualquier cliente puede hacer un `GET` a la ruta estándar `/.well-known/agent-card.json` y obtener la tarjeta del agente.

Vamos a hacer esa petición HTTP "a mano" para que veas el JSON crudo: nombre, skills y endpoints.

In [10]:
import json

card_url = f"{ESPECIALISTA_URL}/.well-known/agent-card.json"
card_json = httpx.get(card_url, timeout=10.0).json()

print(f"GET {card_url}\n")
print(json.dumps(card_json, indent=2, ensure_ascii=False))

GET http://127.0.0.1:8001/.well-known/agent-card.json

{
  "name": "Experto en One Piece",
  "description": "Agente especialista que responde preguntas sobre personajes de One Piece.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:8001/",
      "protocolBinding": "JSONRPC"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "consultar_personaje",
      "name": "Consultar personaje de One Piece",
      "description": "Dado el nombre de un personaje de One Piece, devuelve información sobre su tripulación, recompensa y fruta del diablo.",
      "tags": [
        "one-piece",
        "anime",
        "personajes"
      ],
      "examples": [
        "Dame info de Luffy",
        "¿Qué sabes de Zoro?"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3",
  "url": "http://127.0.0.1:8001/"
}


El JSON anterior es la **Agent Card**. Fíjate en los campos clave:

- `name` y `description`: identidad del agente.
- `skills`: la habilidad `consultar_personaje` con su descripción y ejemplos.
- `supported_interfaces`: la lista de transportes y endpoints donde delegar tareas (aquí, JSON-RPC).
- `capabilities`: aquí `streaming: true`.

Un orquestador lee exactamente esto para decidir si el agente le sirve. **No necesita conocer su código interno**: solo su contrato público.

## 5. El orquestador: cliente A2A que delega una tarea

El **orquestador** es el segundo agente. Su trabajo:

1. Descubrir al especialista leyendo su Agent Card (con `A2ACardResolver`).
2. Comprobar que tiene la skill que necesita.
3. Crear un cliente A2A y **delegarle una tarea**.
4. Escuchar los eventos y recoger el artifact de respuesta.

Primero resolvemos la Agent Card con el SDK (en vez de a mano) y revisamos sus skills.

In [11]:
from a2a.client import A2ACardResolver

async def descubrir_especialista() -> AgentCard:
    async with httpx.AsyncClient() as http_client:
        resolver = A2ACardResolver(httpx_client=http_client, base_url=ESPECIALISTA_URL)
        return await resolver.get_agent_card()


card = asyncio.get_event_loop().run_until_complete(descubrir_especialista())

print(f"[orquestador] agente descubierto: {card.name}")
for s in card.skills:
    print(f"  - skill '{s.id}': {s.description}")

# Decisión del orquestador: ¿hay una skill que encaja?
necesito = "consultar_personaje"
encaja = any(s.id == necesito for s in card.skills)
print(f"\n[orquestador] necesito '{necesito}' -> encaja: {encaja}")

[orquestador] agente descubierto: Experto en One Piece
  - skill 'consultar_personaje': Dado el nombre de un personaje de One Piece, devuelve información sobre su tripulación, recompensa y fruta del diablo.

[orquestador] necesito 'consultar_personaje' -> encaja: True


### Delegar la tarea

Confirmado que el especialista encaja, el orquestador crea un cliente con `ClientFactory` y le envía un mensaje dentro de un `SendMessageRequest`. El SDK gestiona por dentro la creación de la **task**: le asigna un `task_id`, emite eventos de estado y entrega el artifact final.

`send_message` devuelve un stream de eventos. Cada evento trae uno de estos campos: `task` (la tarea inicial), `status_update` (cambio de estado), `artifact_update` (un artifact) o `message` (respuesta directa). Recorremos los eventos y nos quedamos con el texto del artifact.

In [12]:
from a2a.client import ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, SendMessageRequest

# httpx corta a los 5 s por defecto. El especialista llama a un LLM dentro de
# su execute() y eso tarda más; además el stream queda en silencio mientras el
# modelo piensa. Damos un timeout amplio para que no se corte la tarea.
TIMEOUT_A2A = 120.0


def construir_mensaje(texto: str) -> Message:
    # En a2a 1.0 el Part lleva el texto directo y el Role es ROLE_USER.
    return Message(
        role=Role.ROLE_USER,
        message_id=uuid.uuid4().hex,
        parts=[Part(text=texto)],
    )


async def delegar_tarea(card: AgentCard, texto: str) -> dict:
    """El orquestador envía una tarea al especialista y recoge la respuesta."""
    async with httpx.AsyncClient(timeout=TIMEOUT_A2A) as http_client:
        factory = ClientFactory(ClientConfig(httpx_client=http_client, streaming=True))
        client = factory.create(card)

        print(f"[orquestador] delegando tarea: {texto!r}")
        resultado = {"task_id": None, "estados": [], "respuesta": None}

        # send_message recibe un SendMessageRequest y devuelve un stream de
        # eventos. Cada evento (StreamResponse) trae uno de estos campos:
        # task, status_update, artifact_update o message.
        peticion = SendMessageRequest(message=construir_mensaje(texto))
        async for evento in client.send_message(peticion):
            if evento.HasField("task"):
                task = evento.task
                resultado["task_id"] = task.id
                resultado["estados"].append(TaskState.Name(task.status.state))
            elif evento.HasField("status_update"):
                estado = evento.status_update.status.state
                resultado["estados"].append(TaskState.Name(estado))
            elif evento.HasField("artifact_update"):
                for part in evento.artifact_update.artifact.parts:
                    if part.HasField("text"):
                        resultado["respuesta"] = part.text
            elif evento.HasField("message"):
                # Respuesta directa tipo Message (sin tarea).
                for part in evento.message.parts:
                    if part.HasField("text"):
                        resultado["respuesta"] = part.text
        return resultado


salida = asyncio.get_event_loop().run_until_complete(
    delegar_tarea(card, "Dame info de Luffy")
)

print(f"\n[orquestador] task_id: {salida['task_id']}")
print(f"[orquestador] estados de la tarea: {salida['estados']}")

[orquestador] delegando tarea: 'Dame info de Luffy'
  [especialista] tarea recibida: 'Dame info de Luffy'


  [especialista] tarea completada

[orquestador] task_id: c25034c7-5c37-45f9-8974-bf5e3492dfdf
[orquestador] estados de la tarea: ['TASK_STATE_SUBMITTED', 'TASK_STATE_WORKING', 'TASK_STATE_COMPLETED']


## 6. La respuesta llega al orquestador

El especialista terminó su tarea y publicó un artifact. El orquestador lo ha recibido al recorrer los eventos. Veamos el intercambio completo.

In [13]:
print("=" * 60)
print("INTERCAMBIO A2A COMPLETO")
print("=" * 60)
print(f"Orquestador  -> especialista : 'Dame info de Luffy'")
print(f"Task ID                      : {salida['task_id']}")
print(f"Estados de la tarea          : {' -> '.join(salida['estados'])}")
print("-" * 60)
print("Especialista -> orquestador (artifact):\n")
print(salida["respuesta"])
print("=" * 60)

INTERCAMBIO A2A COMPLETO
Orquestador  -> especialista : 'Dame info de Luffy'
Task ID                      : c25034c7-5c37-45f9-8974-bf5e3492dfdf
Estados de la tarea          : TASK_STATE_SUBMITTED -> TASK_STATE_WORKING -> TASK_STATE_COMPLETED
------------------------------------------------------------
Especialista -> orquestador (artifact):

Monkey D Luffy
Tripulación: The Chapeau de Paille crew
Recompensa: 3.000.000.000
Fruta del diablo: Hito Hito no Mi, Nika model
Fuente: api-onepiece.com


## 7. (Opcional) Human-in-the-loop con `input_required`

No todas las tareas se completan a la primera. A veces el agente necesita **más información** de quien la pidió. El protocolo A2A define el estado `input_required`: la tarea queda en pausa esperando una respuesta humana o de otro agente.

Creamos una variante del especialista que, si no recibe nombre de personaje, en vez de fallar pide una aclaración.

In [14]:
class ExpertoConPreguntasExecutor(AgentExecutor):
    """Variante del especialista que pide datos si el mensaje es ambiguo."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Publicamos primero la Task, igual que en el especialista básico.
        task = context.current_task or new_task_from_user_message(context.message)
        await event_queue.enqueue_event(task)

        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.start_work()

        pregunta = context.get_user_input()
        tiene_nombre = any(w[:1].isupper() for w in pregunta.split())

        if not tiene_nombre:
            # Falta el personaje: pausamos la tarea y pedimos aclaración.
            aviso = "Necesito el nombre de un personaje. ¿Sobre quién quieres saber?"
            await updater.update_status(
                TaskState.TASK_STATE_INPUT_REQUIRED,
                message=updater.new_agent_message([Part(text=aviso)]),
            )
            print("  [especialista] tarea en input_required: faltan datos")
            return

        nombre = extraer_nombre_personaje(pregunta)
        texto = redactar_respuesta(pregunta, buscar_personaje(nombre))
        await updater.add_artifact([Part(text=texto)],
                                   name="respuesta_one_piece")
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task or new_task_from_user_message(context.message)
        updater = TaskUpdater(event_queue, task.id, task.context_id)
        await updater.update_status(TaskState.TASK_STATE_CANCELED)


print("Variante con input_required definida")

Variante con input_required definida


Levantamos esta variante en otro puerto (`8002`) y le enviamos un mensaje **ambiguo**, sin nombre de personaje. Veremos que la tarea termina en estado `input_required` en lugar de `completed`.

In [15]:
PORT_2 = 8002
ESPECIALISTA2_URL = f"http://{HOST}:{PORT_2}"

# Los tipos de a2a 1.0 son protobuf: clonamos con CopyFrom y ajustamos campos.
from a2a.types import AgentCard as _AgentCard

agent_card_v2 = _AgentCard()
agent_card_v2.CopyFrom(agent_card_especialista)
agent_card_v2.name = "Experto en One Piece (con preguntas)"
del agent_card_v2.supported_interfaces[:]
agent_card_v2.supported_interfaces.append(
    AgentInterface(protocol_binding="JSONRPC", url=ESPECIALISTA2_URL + "/")
)

handler_v2 = DefaultRequestHandler(
    agent_executor=ExpertoConPreguntasExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=agent_card_v2,
)

rutas_v2 = []
rutas_v2.extend(create_agent_card_routes(agent_card_v2))
rutas_v2.extend(create_jsonrpc_routes(handler_v2, rpc_url="/"))
app_v2 = Starlette(routes=rutas_v2)


servidor_v2 = uvicorn.Server(
    uvicorn.Config(app_v2, host=HOST, port=PORT_2, log_level="warning")
)


def arrancar_servidor_v2():
    servidor_v2.run()


server_thread_v2 = threading.Thread(target=arrancar_servidor_v2, daemon=True)
server_thread_v2.start()
if not _esperar_servidor(ESPECIALISTA2_URL):
    raise RuntimeError("El segundo especialista no respondió a tiempo.")
print(f"Segundo especialista escuchando en {ESPECIALISTA2_URL}")

Segundo especialista escuchando en http://127.0.0.1:8002


In [16]:
async def probar_input_required() -> dict:
    # Mismo timeout amplio que en delegar_tarea: el especialista llama a un LLM.
    async with httpx.AsyncClient(timeout=TIMEOUT_A2A) as http_client:
        resolver = A2ACardResolver(httpx_client=http_client, base_url=ESPECIALISTA2_URL)
        card_v2 = await resolver.get_agent_card()
        factory = ClientFactory(ClientConfig(httpx_client=http_client, streaming=True))
        client = factory.create(card_v2)

        # Mensaje ambiguo: sin nombre de personaje.
        estado_final, aviso = None, None
        peticion = SendMessageRequest(message=construir_mensaje("dame info"))
        async for evento in client.send_message(peticion):
            if evento.HasField("task"):
                estado_final = TaskState.Name(evento.task.status.state)
            elif evento.HasField("status_update"):
                status = evento.status_update.status
                estado_final = TaskState.Name(status.state)
                # El agente adjunta un mensaje pidiendo la aclaración.
                if status.HasField("message"):
                    for part in status.message.parts:
                        if part.HasField("text"):
                            aviso = part.text
        return {"estado": estado_final, "aviso": aviso}


res = asyncio.get_event_loop().run_until_complete(probar_input_required())

print(f"[orquestador] estado final de la tarea: {res['estado']}")
print(f"[orquestador] el agente pide: {res['aviso']}")
print("\nAquí es donde entraría un humano (o el orquestador) a aportar el dato que falta.")

  [especialista] tarea en input_required: faltan datos
[orquestador] estado final de la tarea: TASK_STATE_INPUT_REQUIRED
[orquestador] el agente pide: Necesito el nombre de un personaje. ¿Sobre quién quieres saber?

Aquí es donde entraría un humano (o el orquestador) a aportar el dato que falta.


## 8. Recap: el handshake de A2A

Has visto, de principio a fin, como dos agentes colaboran con A2A:

1. **Discovery** · El especialista publica su **Agent Card** en `/.well-known/agent-card.json`. El orquestador la lee con `A2ACardResolver` y descubre la skill `consultar_personaje`.
2. **Delegación** · El orquestador crea un cliente con `ClientFactory` y envía un mensaje. El SDK genera una **task** con su `task_id` y emite eventos de estado: `working` -> `completed`.
3. **Respuesta** · El especialista ejecuta su skill (API de One Piece + LLM) y publica un **artifact**. El orquestador lo recibe sin conocer el código interno del especialista.
4. **Human-in-the-loop** · Cuando faltan datos, la tarea termina en `input_required` y queda a la espera de una aclaración.

Ideas para llevarte:

- A2A desacopla agentes: solo comparten el **contrato público** (la Agent Card).
- Un agente puede ser **servidor** (especialista) y otro **cliente** (orquestador), o ambas cosas a la vez.
- El protocolo es agnóstico del framework: el especialista podría estar hecho con LangChain, ADK o lo que sea.

### Errores comunes

- **El servidor bloquea el notebook**: hay que lanzar uvicorn en un hilo de fondo y aplicar `nest_asyncio.apply()`.
- **`Connection refused`**: el servidor aún no está listo. Por eso esperamos con un health-check (`_esperar_servidor`) en vez de un `time.sleep` fijo.
- **Puerto ocupado**: si re-ejecutas el notebook sin apagar los servidores, el puerto sigue en uso. Ejecuta la sección 9 (o reinicia el kernel) antes de volver a empezar.

### Ejercicios

1. Cambia la pregunta de la sección 5 (`delegar_tarea`) por "¿Qué sabes de Zoro?" y observa el nuevo artifact.
2. Añade un personaje nuevo al diccionario `ONE_PIECE_FALLBACK` y consúltalo.
3. Crea un **tercer agente** (otro especialista, por ejemplo de tripulaciones) y haz que el orquestador elija a cuál delegar según la skill de cada Agent Card.

## 9. Apagar los servidores

Los dos especialistas siguen escuchando en `8001` y `8002` mientras el kernel esté vivo. Antes de cerrar el notebook (o de volver a ejecutarlo de arriba a abajo) los apagamos limpiamente: `should_exit = True` le pide a uvicorn que termine y `join()` espera a que el hilo acabe. Así los puertos quedan libres.


In [17]:
import logging

# La tarea de la sección 7 se quedó abierta en input_required: al apagar, el SDK
# avisa de que su cola de eventos ya no tiene dispatcher. Es esperado; lo silenciamos.
logging.getLogger("a2a.server.events.event_queue_v2").setLevel(logging.ERROR)

# Pedimos a cada uvicorn que termine y esperamos a que su hilo acabe.
for nombre, servidor, hilo in [
    ("especialista (8001)", servidor_especialista, server_thread),
    ("especialista con preguntas (8002)", servidor_v2, server_thread_v2),
]:
    servidor.should_exit = True
    hilo.join(timeout=10)
    print(f"Servidor {nombre}: {'apagado' if not hilo.is_alive() else 'sigue vivo, reinicia el kernel'}")


Servidor especialista (8001): apagado


Servidor especialista con preguntas (8002): apagado
